# Recitaton 0.5
## Tensors, Einstein Summation Notation, Einsum, and Tensordot

### Learning Objective
The goal of recitation 0.5 is to get you up to speed with the einsum and tensordot tensor operators in 
numpy and pytorch by providing an overview of einstein summation notation, tensor conventions, and basic tensor operations. 

Topics: 
* 1 What is a Tensor? 
  - 1.1 Common Tensor Conventions
  - 1.2 Imagining and Conceptualizing Tensors
* 2 Einstein Summation Notation
  - 2.1 matrix multiplication
  - 2.2 dot product
* 3 Einsum
  - 3.1 documentation
  - 3.2 matrix multiplication
  - 3.3 dot product
  - 3.4 hadamard product
  - 3.5 tranpose
  - 3.6 outerproduct
  - 3.7 trace

* 4 Tensordot
  - 4.1 documentation
  - 4.2 matrix multiplication
  - 4.3 dot product
  - 4.4 multi-dimensional dot product

## 1 What is a Tensor?

A tensor is a datastructure that can be thought of as an $n$ dimensional array. The tensor is one of the fundamental units of deep learning. In pytorch tensors are represented by torch.tensor objects and in numpy tensors are represented by np.ndarray objects. Different deep learning libraries have different implementations of tensors.

### 1.1 Common Tensor Conventions

Tensors typically represent data or model parameters in the context of deep learning. Data and model parameters typically have standard tensor dimension layouts. When processing data in batches, the outermost dimension of a tensor is almost always the batch dimension. When refering to a tensors dimensions, the 0th dimension is always the outermost dimension and the nth dimension is always the innermost dimension. It is helpful to think of the innermost dimension as storing the actual data and all outer dimensions simply organizing this data. This convention is followed when indexing shape objects in both numpy and pytorch. 

It is worthy to note that in contrast to typical matrices, where vectors usually make up the columns of a matrix, in deep learning vectors always form the rows of a matrix or tensor. This leads to differing mathematical and computational notations and implementations for matrix multiplication and other common matrix and tensor operations. 


#### 1.1.1 Vector Data

Vectors are simply 1D tensors. Their shape is denoted as (N) where N is the number of elements present in the vector. If we want to represent a batch of vectors we get a 2D tensor with shape (B, N) wher B is the batch dimension and N is the number of elements in each vector. 

#### 1.1.2 Image Data

A single image is typically represented by a 3D tensor with shape (C, H, W) where C represents the number of channels, H is the height of the image in pixels, and W is the width of the image in pixels. C is used when images are RGB in which case there a typically 3 channels in an image. If C = 1, then the image is greyscale and becomes 2D. A batch of images has shape (B, C, H, W) where B is the batch dimension.

#### 1.1.3 Sequence Data

There are many types of sequence data. For example a sequence of images makes a video, a sequence of normalized real numbers makes audio, and a sequence of real numbers makes a stock signal. Sequence data is represented in 2 standard ways. Either the data is represented in a batch first or sequence first manner. Batch first representation is of shape (B, L, ...) where B is the batch dimension, L is the number of timesteps, and ... represents n-dimensinoal per time step data. Sequence first representation simply flips the batch and sequence dimension creating a shape of (L, B, ...). Seqence first storage is common in RNNs and batch first storage is common in Transformers. 


### 1.2 Imagining and Conceptualizing Tensorsand Tips

Sometimes when you are creating and training models it can be hard to envision what is happening. Consequently, I recommend annotating all functions and layers with the input and output shape of their transformation. The most simple way to do this is demonstrated below. 

Furthermore, I recomend attempting to envision transformations in your head. While you are practicing this it helps to draw out tensor shpaes on a white board to assist your mind in visualizing. 

In [ ]:
def layer(): 
    """
    in: (B, H, W) -> out: (B, C, H, W)
    """

    ### layer operatiosn performing the given transformation

## 2 Einstein Summation Notation

In order to understnad einsum and tensordot we must first undnerstand einstein summation notation. This convention creates a more concies notation for representing matrix operations. It is easiest to learn via an example you have likely seen before. We will first examine matrix multiplication to get a grip on einstein summation notation. 

### 2.1 Matrix Multiplication 

Lets consider stanndard matrix multiplication. Let $A \in \mathbb{R}^{i \times j}$ and $B \in \mathbb{R}^{j \times k}$ be matrices. Standard matrix mutliplication is defined as: 

$$
C_{i,k} = \Sigma_{j} A_{i,j} B_{j, k} \equiv A_{i,j} B_{j,k}
$$

We can see then that: 

$$
C_{i,k} = A_{i,j} B_{j,k}
$$

The second definition is known as einstein summation notation. Instead of including the sum we omit it and infer that the matrices are summed over their shared dimension $j$. It is helpful to think of how this computation is carried out. First the $j$ dimensions are lined up and mulitplied elementwise. Then a sum is computed over the $j$ dimension and the new element at $C_{i,k}$ is the dot product of the ith $j$ dimensional vector in matrix $A$ with the kth $j$ dimensional vector in matrix $B$.

### 2.2 Dot Product

Lets now consider the einstein summation notation for the dot product. This should feel a little more intuitive after our discussion of matrix multitplication. let $\vec{a}, \vec{b} \in \mathbb{R^i}$. Then one simple definition of the dot product between these vectors is:

$$
s = \Sigma_i \vec{a}_i \vec{b}_i \equiv \vec{a}_i \vec{b}_i
$$

Again by the summation convention it is implied that the two vectors are first elementwise multiplied and then summed over their shared dimension $i$. 

## 3 Einsum

Many matrix libraries include an einsum function that performs tensor operations following the einsum convention. We will now go over the functioning of einsum in numpy. np.einsum() functions similarly to einsum in other libraries and therefore the skills you learn here are transferable to pytorch.

### 3.1 Documentation

The documentation for np.einsum is as follows: 

np.einsum(subscripts, *operands, out=None, optimize=False)

- subscripts is a string containing the subscripts of the tensors to operate
- *operands are the tensors $A$ and $B$ to operate on
- out is an opitional tensor to write the output to
- optimize is set to True in order to utilize numpy optimizations

The subscript string holds the einsum notation that describes the tensor operation. There are two ways to write these strings, explicit mode and implicit mode. Implicit mode uses standard einsum notation with the summation sign omitted. Explicit mode uses an arrow to specify the dimensions of the tensor that results from the operation. It is recomended to use explicit mode. We will examine matrix multiplication once again to get a handle on this operation

Before we look at matrix multiplication though, let us remember the steps of einsum. First matching dimension are aligned and multiplied element wise. Then matching dimensions are summed over or in other words contracted. 

### 3.2 Matrix Mutliplication

In [ ]:
import numpy as np

A = np.ndarray([
    [1,2,1], 
    [3,1,2]
])

B = np.ndarray([
    [1,2],
    [2,1],
    [1,1]
])

"""
Explicit Mode: 
    We can see that the new tensor C has shape ik as denoted in the 
    subscript string. These subscripts can be though of as the subscripts
    in the math notation provided earlier. We see that the j dimension is 
    ommitted on the right side of the arrow as it is summed over. 
"""
C = np.einsum("ij, jk -> ik", A, B)

"""
Implicit Mode
    This performs the same operation as explicit mode but follows the 
    mathematical convention more closely. It is implied that summation 
    is performed over the matching dimension j. 
"""
C = np.einsum("ij,jk", A, B)

print(C)

### 3.3 Dot Product

In [ ]:
import numpy as np 

u = np.ndarray([1,2,1])

v = np.ndarray([2,1,2])

w = np.einsum("i,i -> ", u, v)

print(w)

### 3.4 Hadamard Product (Elementwise Multiplication)

In [ ]:
import numpy as np 

A = np.ndarray([
    [1,2,1],
    [2,1,3]
])

B = np.ndarray([
    [1,2,1],
    [2,1,2]
])

C = np.einsum("ij,ij -> ij", A, B)

print(C)

### 3.5 Tranpose

In [ ]:
import numpy as np

A = np.ndarray([
    [1,2,3],
    [1,2,3],
    [1,2,3]
])

B = np.einsum("ij -> ji", A)

print(B)

### 3.6 Outer Product

In [ ]:
import numpy as np 

u = np.ndarray([1,2,1])

v = np.ndarray([2,1,2, 4])

C = np.einsum("i,j -> ij ", u, v)

print(w)

### 3.7 Trace

In [ ]:
import numpy as np

A = np.ndarray([
    [1,2,3],
    [1,2,3],
    [1,2,3]
])

B = np.einsum("ii ->", A)

print(B)

## 4 Tensordot

Many tensor libraries also include a tensordot operation. This operation is functionally a less general einsum. The primary function of tensordot is to elementwise multiply and contract specified dimensions of a tensor. Unlike einsum you may only contract over specified dimensions. Therefore you may not perform transposes and other operations like hadamard products. It is helpful to think of this operation as the taking the dot product of all combination of the specified axes from matrices A and B. 

### 4.1 Documentation

The documentation for np.tensordot() is as follows: 

np.tensordot(A, B, axes=([a1, ..., an], [b1, ..., bn]))

Where a and b are tensors and a1, ..., an and b1, ..., bn are axes of matching shape that are to be elementwise multiplied and contracted. 

### 4.2 Matrix Multiplication

In order to perform matrix multiplication we must first figure out which axes to line up. From our previous study of einsum, we know that the first (innermost) and zeroth (outermost) dimension of matrices $A$ and $B$ are elementwise multiplied and contracted in order to perform matrix multiplication. Therefore the axes we must use in tensor dot are 0 and 1 respectively. Notice that the axes not specified in the operation remain. Any axis not contracted will remain in your tensor. 

In [ ]:
import numpy as np

A = np.ndarray([
    [1,2,1],
    [2,1,3]
])

B = np.ndarray([
    [1,2,1],
    [2,1,2]
])

C = np.tensordot(A, B, axes=([1], [0]))

print(C)


### 4.3 Dot Product

In order to perform the dot product we simply select the first and only dimension for both axes. This performs elementwise multiplication and summataion over the first dimension which is exactly the definition of a dot product. 

In [ ]:
import numpy as np

u = np.ndarray([1,2,1])

v = np.ndarray([2,1,2])

w = np.tensordot(u, v, axes=([1], [1]))

print(w)


### 4.4 Multi-Dimensional Dot Product

For our final example lets imagine that we have two batches of matrices. In otherwords we have two tensors $A$ and $B$ of shape (B, I, J) where B is the batch dimension and I and J are the dimensions of each individual matrix. We want to compute the elementwise sum of all of the matrices with each other. Therefore we can take the tensordot over the first and second dimensions of the matrices. This will leave us with a tensor of shape (B, B) holding all the matrix dot products (frobenius products) of all combinations of the matrices stored in our two tensors. 

In [ ]:
import numpy as np 

rng = np.random.default_rng()

#shape is (3,2,2)
A = np.rng.random((3,2,2))

#shape is (3,2,2)
B = np.rng.random((3,2,2))

#shape is (3, 3)
C = np.tensordot(A, B, axes=([1,2], [1,2]))

print(C)

## Summary 

In  this recitation we have learned about einstein summation notation, einsum, tesnordot, and fundamental tensor operations. You are greatly encouraged to go try these tools out for yourself (and you will have to if you intend to complete 11785). A good way to practice is to reimplement the simple code I have provided here from scratch. If you can do this and reason through your approach you are in good shape. Good luck!